# Notebook 03: Data Preparation

**Purpose:** Split data into train/val/test, convert annotations to DocuSpend schema.

In [ ]:
import sys
sys.path.append('./docuspend/src')

from pathlib import Path
import json
import os
from annotation_converter import AnnotationConverter
from dataset import DatasetSplitter

base_dir = Path("/content/docuspend") if os.path.exists("/content") else Path("./docuspend")
print(f"Working with base directory: {base_dir}")

In [ ]:
# Step 1: Load downloaded images and annotations
raw_all_dir = base_dir / "data/raw/all"
image_files = sorted(raw_all_dir.glob("receipt_*.jpg"))
print(f"Found {len(image_files)} images")

# Load CORU annotations
with open(raw_all_dir / "coru_annotations_raw.json", 'r') as f:
    coru_annotations = json.load(f)
print(f"Loaded {len(coru_annotations)} annotations")

In [ ]:
# Step 2: Train/Val/Test split (80/10/10)
import random

random.seed(42)
filenames = [f.name for f in image_files]
random.shuffle(filenames)

total = len(filenames)
train_size = int(total * 0.8)
val_size = int(total * 0.1)

train_files = filenames[:train_size]
val_files = filenames[train_size:train_size + val_size]
test_files = filenames[train_size + val_size:]

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

In [ ]:
# Step 3: Distribute images to split directories
import shutil

splits = {"train": train_files, "val": val_files, "test": test_files}

for split, files in splits.items():
    split_dir = base_dir / f"data/raw/{split}"
    split_dir.mkdir(parents=True, exist_ok=True)
    
    for filename in files:
        src = raw_all_dir / filename
        dst = split_dir / filename
        shutil.copy2(src, dst)
    
    print(f"✓ Copied {len(files)} images to {split}")

In [ ]:
# Step 4: Convert CORU annotations to DocuSpend schema
converter = AnnotationConverter()
split_annotations = {"train": [], "val": [], "test": []}

for split, files in splits.items():
    ann_dir = base_dir / f"data/annotations/{split}"
    ann_dir.mkdir(parents=True, exist_ok=True)
    
    for filename in files:
        # Convert annotation
        coru_ann = coru_annotations.get(filename, {})
        docuspend_ann = converter.convert_coru_to_docuspend(coru_ann, filename)
        
        if docuspend_ann:
            # Save annotation
            ann_file = ann_dir / f"{Path(filename).stem}.json"
            with open(ann_file, 'w') as f:
                json.dump(docuspend_ann, f, indent=2)
            split_annotations[split].append(filename)
    
    print(f"✓ Converted {len(split_annotations[split])} annotations for {split}")

In [ ]:
# Step 5: Validation checks
from dataset import AnnotationValidator

print("Validating annotations...")
for split in ["train", "val", "test"]:
    ann_dir = base_dir / f"data/annotations/{split}"
    stats = AnnotationValidator.validate_dataset(str(ann_dir))
    print(f"{split}: {stats['valid']}/{stats['total_files']} valid")

In [ ]:
# Step 6 & 7: Generate split summary
split_summary = {
    "total_samples": total,
    "train": {"count": len(train_files)},
    "val": {"count": len(val_files)},
    "test": {"count": len(test_files)},
    "annotation_schema": "DocuSpend v1",
    "categories": ["groceries", "dining", "transport", "utilities", "shopping", "healthcare", "entertainment", "other"]
}

splits_dir = base_dir / "data/splits"
splits_dir.mkdir(parents=True, exist_ok=True)
with open(splits_dir / "split_summary.json", 'w') as f:
    json.dump(split_summary, f, indent=2)

print("Split Summary:")
for k, v in split_summary.items():
    print(f"  {k}: {v}")

In [ ]:
# Step 8: Output status report
print("="*60)
print("DATA PREPARATION COMPLETE")
print("="*60)
print(f"✅ Train: {len(train_files)} images")
print(f"✅ Val: {len(val_files)} images")
print(f"✅ Test: {len(test_files)} images")
print(f"✅ Annotations converted to DocuSpend schema")
print(f"\nNext: Run 04_preprocessing.ipynb")